# 02 — Stage 1 Detector Training & Analysis

**YOLOv8s** fine-tuned to detect `pothole` and `traffic_light` in dashcam frames.

---

## How to use this notebook

| Goal | What to do |
|---|---|
| First training run | Run all cells top to bottom |
| Already trained, just view results | Run cells 1–2, then jump to **Section 4** |
| Re-train with different settings | Change values in **Section 2**, re-run Section 3 |
| Improve a bad result | Read the hints under each chart, adjust Section 2, re-run |


## Section 1 — Imports & GPU Check

In [ ]:
import sys, time, shutil
sys.path.insert(0, '../src')

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd
from pathlib import Path
from ultralytics import YOLO
import torch

BASE_DIR   = Path('..').resolve()
MODELS_DIR = BASE_DIR / 'models'
MODELS_DIR.mkdir(exist_ok=True)

print('=' * 55)
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'  GPU  : {p.name}')
    print(f'  VRAM : {p.total_memory/1e9:.1f} GB')
else:
    print('  No GPU — will train on CPU (very slow)')

try:
    import psutil
    print(f'  RAM  : {psutil.virtual_memory().total/1e9:.1f} GB')
except ImportError:
    pass
print('=' * 55)

## Section 2 — ⚙️ Configure Training Parameters

**Change values here before running training.**
Everything else in the notebook is automatic.


In [ ]:
# ┌─────────────────────────────────────────────────────────────────┐
# │            CHANGE THESE VALUES TO TUNE TRAINING                 │
# └─────────────────────────────────────────────────────────────────┘

USE_AUG      = True    # True  = train on 79k weather-augmented images
                       # False = train on 26k original images (3x faster)

EPOCHS       = 25      # Set to 25 for a quick test, 100+ for full training
                       # None = auto (scales with fraction, ~285 for aug data)

FRACTION     = None    # Fraction of dataset per epoch
                       # None  = auto (~0.35 for aug, 1.0 for original)
                       # 1.0   = use full dataset every epoch (slower)
                       # 0.35  = use 35% per epoch (3x faster, same total exposure)

IMGSZ        = 640     # Image size. 640 = standard. 832 = better for small objects.

BATCH        = None    # None = auto from VRAM (safe)
                       # Set a number (e.g. 64) only if auto-batch causes OOM

CONF_THRESH  = 0.40    # Confidence threshold for validation predictions display

# ─────────────────────────────────────────────────────────────────
# Auto-preview what will run
from train import _gpu_info, _auto_batch, _auto_fraction, _count_images

_, n_gpus, vram_gb = _gpu_info()
auto_batch = _auto_batch(vram_gb, IMGSZ)
batch_used = BATCH if BATCH else auto_batch

aug_yaml  = BASE_DIR / 'data/processed/detector_yolo_aug/dataset.yaml'
orig_yaml = BASE_DIR / 'data/processed/detector_yolo/dataset.yaml'
yaml_used = aug_yaml if USE_AUG and aug_yaml.exists() else orig_yaml
n_images  = _count_images(yaml_used)

frac_used = FRACTION if FRACTION else _auto_fraction(n_images)
from math import ceil
epoch_scale = max(1.0, 1.0 / frac_used)
auto_epochs = min(300, int(100 * epoch_scale))
epochs_used = EPOCHS if EPOCHS else auto_epochs

steps_per_epoch = int(n_images * frac_used) // batch_used
est_min = steps_per_epoch * 0.338 / 60

print('\n  Training preview:')
print(f'    Dataset      : {"augmented" if USE_AUG else "original"}  ({n_images:,} images)')
print(f'    Images/epoch : {int(n_images * frac_used):,}  (fraction={frac_used})')
print(f'    Epochs       : {epochs_used}')
print(f'    Batch size   : {batch_used}  (VRAM={vram_gb:.1f}GB)')
print(f'    Image size   : {IMGSZ}px')
print(f'    Est/epoch    : ~{est_min:.1f} min')
print(f'    Est total    : ~{est_min * min(epochs_used, 60):.0f}–{est_min * epochs_used:.0f} min')

## Section 3 — Train

> **Skip this section if you already trained.** Jump straight to Section 4.

Training prints live progress every epoch. Watch `mAP50` — it should rise steadily.
If it stops improving for many epochs, the model has converged and you can stop early.


In [ ]:
from train import train_detector

t0 = time.time()

results = train_detector(
    use_aug        = USE_AUG,
    imgsz          = IMGSZ,
    fraction       = FRACTION,
    epochs_override = EPOCHS,
    batch_override  = BATCH,
    run_eval       = False,
)

elapsed = time.time() - t0
print(f'\n  Training done in {elapsed/60:.1f} min')

# Copy best weights to models/
best = BASE_DIR / 'runs/detector/weights/best.pt'
dest = MODELS_DIR / 'detector_model.pt'
if best.exists():
    shutil.copy2(best, dest)
    print(f'  Saved → {dest}')

---
## Section 4 — Results & Analysis

Run from here if you already trained. These cells load the saved results and model.

In [ ]:
# Copy weights if training was done outside this notebook
best = BASE_DIR / 'runs/detector/weights/best.pt'
dest = MODELS_DIR / 'detector_model.pt'
if best.exists() and not dest.exists():
    shutil.copy2(best, dest)
    print(f'  Copied weights → {dest}')

results_csv = BASE_DIR / 'runs/detector/results.csv'
if not results_csv.exists():
    print('  No results.csv found. Run Section 3 first.')
else:
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    print(f'  Results loaded: {len(df)} epochs')

    # Find best epoch
    map_col = 'metrics/mAP50(B)'
    if map_col in df.columns:
        best_ep  = df[map_col].idxmax()
        best_map = df[map_col].max()
        last_map = df[map_col].iloc[-1]
        trend    = df[map_col].iloc[-5:].diff().mean()
        print(f'\n  Best mAP@0.5  : {best_map:.4f}  at epoch {best_ep}')
        print(f'  Last mAP@0.5  : {last_map:.4f}')
        print(f'  Last 5ep trend: {trend:+.4f}  ', end='')
        if trend > 0.002:
            print('↑ Still improving — consider more epochs')
        elif trend > -0.001:
            print('→ Converged')
        else:
            print('↓ Declining — may be overfitting')

### 4a — Training Curves

**What to look for:**
- Loss curves should fall smoothly and level off
- `val` loss should track `train` loss — if val loss rises while train falls → **overfitting**
- `mAP50` should rise and plateau — the gold line marks the best epoch saved
- If mAP50 is still rising at the last epoch → **need more epochs**


In [ ]:
if 'df' not in dir() or df is None:
    print('Run the cell above first'); raise SystemExit

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
plot_pairs = [
    ('train/box_loss',       'val/box_loss',         'Box Loss'),
    ('train/cls_loss',       'val/cls_loss',          'Class Loss'),
    ('train/dfl_loss',       'val/dfl_loss',          'DFL Loss'),
    ('metrics/precision(B)', None,                    'Precision'),
    ('metrics/recall(B)',    None,                    'Recall'),
    ('metrics/mAP50(B)',     'metrics/mAP50-95(B)',  'mAP@0.5 / mAP@0.5:0.95'),
]
for ax, (tc, vc, title) in zip(axes.flatten(), plot_pairs):
    if tc in df.columns:
        ax.plot(df['epoch'], df[tc], label='train', color='#3498db', linewidth=1.5)
    if vc and vc in df.columns:
        ax.plot(df['epoch'], df[vc], label='val',   color='#e74c3c',
                linestyle='--', linewidth=1.5)
    ax.set_title(title, fontsize=11); ax.set_xlabel('Epoch')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

map_col = 'metrics/mAP50(B)'
if map_col in df.columns:
    best_ep  = df[map_col].idxmax()
    best_map = df[map_col].max()
    axes[1][2].axvline(best_ep, color='gold', linestyle=':', linewidth=2,
                       label=f'Best ep {best_ep}  mAP={best_map:.3f}')
    axes[1][2].legend(fontsize=8)

plt.suptitle('Stage 1 Detector — Training Curves', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

### 4b — Validation Metrics

**What to look for:**
- `mAP@0.5 > 0.70` is good for a 2-class detector on this dataset
- `mAP@0.5 < 0.50` after 25 epochs → normal, need more training
- Large gap between Precision and Recall → adjust `conf_thresh` at inference time


In [ ]:
model_path = MODELS_DIR / 'detector_model.pt'
if not model_path.exists():
    print(f'  Model not found: {model_path}')
    print('  Run Section 3, or: cp runs/detector/weights/best.pt models/detector_model.pt')
else:
    model    = YOLO(str(model_path))
    metrics  = model.val(data=str(BASE_DIR / 'data/processed/detector_yolo/dataset.yaml'),
                         verbose=False)
    map50    = metrics.box.map50
    map5095  = metrics.box.map
    prec     = metrics.box.mp
    rec      = metrics.box.mr

    # Traffic light summary
    print(f'\n  mAP@0.5      : {map50:.4f}', end='  ')
    if   map50 >= 0.75: print('Excellent')
    elif map50 >= 0.60: print('Good — ready for pipeline testing')
    elif map50 >= 0.45: print('Acceptable — more training will help')
    else:               print('Needs more training')

    print(f'  mAP@0.5:0.95 : {map5095:.4f}')
    print(f'  Precision    : {prec:.4f}  (of predicted boxes, how many are correct)')
    print(f'  Recall       : {rec:.4f}  (of real objects, how many are found)')

### 4c — Per-Class AP

**What to look for:**
- Both classes should have similar AP — if one is much lower, that class needs more data or augmentation


In [ ]:
if 'metrics' not in dir():
    print('Run the validation cell above first')
elif hasattr(metrics.box, 'ap50'):
    class_names = ['pothole', 'traffic_light']
    colors      = ['#e74c3c', '#2ecc71']
    fig, ax = plt.subplots(figsize=(6, 4))
    bars = ax.bar(class_names, metrics.box.ap50, color=colors, width=0.4)
    for bar, v in zip(bars, metrics.box.ap50):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.01,
                f'{v:.3f}', ha='center', fontweight='bold', fontsize=12)
    ax.set_ylim(0, 1.1); ax.set_title('AP@0.5 per Class'); ax.set_ylabel('AP@0.5')
    ax.axhline(0.5, color='gray', linestyle=':', alpha=0.5)
    plt.tight_layout(); plt.show()

    diff = abs(metrics.box.ap50[0] - metrics.box.ap50[1])
    if diff > 0.15:
        weaker = class_names[metrics.box.ap50.argmin()]
        print(f'  Imbalance detected: {weaker} is weaker by {diff:.2f}')
        print(f'  → Consider adding more {weaker} training images or adjusting augmentation')

### 4d — Sample Predictions on Validation Images

**What to look for:**
- Correct boxes around potholes (red) and traffic lights (green)
- Confidence scores — low scores (<50%) suggest the model is uncertain
- Missed detections or wrong class labels indicate more training is needed


In [ ]:
if 'model' not in dir():
    model = YOLO(str(MODELS_DIR / 'detector_model.pt'))

val_dir    = BASE_DIR / 'data/processed/detector_yolo/images/val'
val_imgs   = sorted(val_dir.glob('*'))[:6]
box_colors = {0: (255, 80, 80), 1: (80, 220, 80)}
class_names = ['pothole', 'traffic_light']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, img_path in zip(axes.flatten(), val_imgs):
    frame = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    preds = model(img_path, conf=CONF_THRESH, verbose=False)
    n_det = len(preds[0].boxes)
    for box in preds[0].boxes:
        x1,y1,x2,y2 = [int(v) for v in box.xyxy[0]]
        cls  = int(box.cls[0]); conf = float(box.conf[0])
        c    = tuple(v/255 for v in box_colors.get(cls, (200,200,200)))
        ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,
                                       linewidth=2,edgecolor=c,facecolor='none'))
        ax.text(x1, max(y1-4,0), f'{class_names[cls]} {conf:.0%}',
                color=c, fontsize=7, fontweight='bold')
    ax.imshow(frame); ax.axis('off')
    ax.set_title(f'{img_path.name[:28]}  [{n_det} det]', fontsize=7)

plt.suptitle(f'Val Predictions  conf≥{CONF_THRESH}  (red=pothole  green=traffic_light)',
             fontsize=12)
plt.tight_layout(); plt.show()

### 4e — Confusion Matrix

**What to look for:**
- Diagonal should be dark (correct predictions)
- Off-diagonal = misclassification — e.g. pothole predicted as background
- `background` row = false negatives (missed objects)


In [ ]:
candidates = list((BASE_DIR / 'runs/detector').rglob('confusion_matrix_normalized.png'))
if not candidates:
    candidates = list((BASE_DIR / 'runs/detector').rglob('confusion_matrix.png'))
if candidates:
    img = cv2.cvtColor(cv2.imread(str(candidates[0])), cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(7, 6))
    plt.imshow(img); plt.axis('off')
    plt.title('Confusion Matrix — Stage 1 Detector')
    plt.show()
else:
    print('  Confusion matrix not found — it is generated during model.val()')

---
## Section 5 — What to Do Next

| mAP@0.5 result | Recommendation |
|---|---|
| < 0.40 | Normal for 25 epochs. Set `EPOCHS = 100` and re-run Section 3 |
| 0.40 – 0.60 | Decent start. Run with `USE_AUG = True, EPOCHS = None` for full training |
| 0.60 – 0.75 | Good. Pipeline testing will work. Full training gets you to 0.75+ |
| > 0.75 | Excellent. Proceed to Stage 2a severity training (notebook 03) |

If one class AP is much lower than the other → that class needs more data.

If val loss is much higher than train loss → overfitting → add `augment=True` or reduce epochs.


In [ ]:
# Quick guide: print next command based on current mAP
if 'map50' in dir():
    print('Current mAP@0.5:', round(map50, 4))
    print()
    if map50 < 0.50:
        print('  Recommendation: Run more epochs')
        print('  Change:  EPOCHS = 100  (re-run Section 2 + 3)')
    elif map50 < 0.70:
        print('  Recommendation: Full training')
        print('  Change:  EPOCHS = None, FRACTION = None, USE_AUG = True  (re-run Section 2 + 3)')
    else:
        print('  Ready for next step!')
        print('  Open: 03-train-severity.ipynb')